# Paper-1 reproduction: five recordings, four voice segments

This notebook walks through the **full** paper-1 pipeline — pitch
tracking, reliability filtering, three-axis metrics, per-recording
aggregation, type classification, and all six paper figures — across
the entire five-recording corpus.

It is the batch counterpart of [`01_quickstart.ipynb`](01_quickstart.ipynb),
which analyses a single segment.

## Audio assumptions

The paper-1 corpus uses five commercial recordings that **cannot be
redistributed**:

- `ath-1973`  Atherton / London Sinfonietta
- `hul-2012`  Hulburt et al.
- `bou-1961`  Boulez (early)
- `bou-1977`  Boulez (later)
- `her-1991`  Heringer / Sinopoli

To reproduce the paper numbers exactly, place these recordings under
`data/audio/<recording_id>/<piece_id>.wav` (or any extension supported
by `soundfile`). For demonstration the notebook will gracefully skip
recordings whose audio is missing and show whatever subset is
available — useful when only the Stiedry-Wagner 1940 file fetched by
`scripts/fetch_audio.py` is on disk.

See [LEGAL_NOTICE.md](../LEGAL_NOTICE.md) for the legal status of the
fetched demo audio.


## 1. Setup

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf

# Make the package importable when running from the repo without `pip install`.
pkg_path = Path("..").resolve() / "src"
if str(pkg_path) not in sys.path:
    sys.path.insert(0, str(pkg_path))

import sprechstimme_pitch as sp
from sprechstimme_pitch import metrics, pitch, plotting

print(f"sprechstimme_pitch version: {sp.__version__}")

REPO_ROOT = Path("..").resolve()
AUDIO_DIR = REPO_ROOT / "data" / "audio"
METADATA_DIR = REPO_ROOT / "data" / "metadata"
EXPORT_DIR = REPO_ROOT / "outputs" / "paper1"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Configuration

Define which recordings to analyse and where to find their audio. The
default list is the paper-1 corpus; trim it if you only have a subset.


In [ ]:
# Paper-1 corpus identifiers.
RECORDINGS = ["ath-1973", "hul-2012", "bou-1961", "bou-1977", "her-1991"]
PIECE_ID = "07"

# pYIN parameters used in paper 1.
FMIN_HZ = 130.8   # C3
FMAX_HZ = 523.3   # C5
FRAME_LENGTH = 2048
HOP_LENGTH = 256


def find_audio(recording_id: str) -> Path | None:
    """Return the first audio file we can find for this recording.

    Search order:
      1. ``data/audio/<recording_id>/`` (any audio extension)
      2. ``data/audio/`` for a file whose stem starts with ``recording_id``
      3. ``data/audio/`` if exactly one audio file is present
         (so a single fetched demo file is picked up automatically).
    """
    exts = (".wav", ".flac", ".mp3", ".m4a", ".ogg", ".aiff", ".aif")

    sub = AUDIO_DIR / recording_id
    if sub.is_dir():
        for ext in exts:
            for p in sorted(sub.glob(f"*{ext}")):
                return p

    for ext in exts:
        for p in sorted(AUDIO_DIR.glob(f"{recording_id}*{ext}")):
            return p

    all_audio = [p for ext in exts for p in AUDIO_DIR.glob(f"*{ext}")]
    if len(all_audio) == 1:
        return all_audio[0]
    return None


## 3. Load metadata

In [ ]:
segments_df = pd.read_csv(METADATA_DIR / "segments.csv")
score_events_df = pd.read_csv(METADATA_DIR / "score_events.csv")
segment_score_map_df = pd.read_csv(METADATA_DIR / "segment_score_map.csv")

# Restrict to the piece under analysis.
segments_df = segments_df[segments_df["piece_id"].astype(str) == PIECE_ID].copy()
score_events_df = score_events_df[score_events_df["piece_id"].astype(str) == PIECE_ID].copy()
segment_score_map_df = segment_score_map_df.merge(
    segments_df[["recording_id", "segment_id"]],
    on=["recording_id", "segment_id"],
    how="inner",
)

print(f"Recordings in metadata     : {sorted(segments_df['recording_id'].unique())}")
print(f"Voice segments in metadata : {sorted(segments_df['segment_id'].unique())}")
print(f"Score events for piece {PIECE_ID}: {len(score_events_df)} notes")


## 4. Per-segment analysis

For each ``(recording, segment)`` pair we:

1. Load the audio slice for that segment.
2. Run pYIN on the slice.
3. For every score event mapped to that segment, derive
   ``voiced_ratio``, ``f0_iqr_cent``, the median observed pitch, and
   the resulting ``is_pyin_unreliable`` flag.
4. Pass the per-note arrays to
   :func:`sprechstimme_pitch.metrics.compute_three_axis_metrics`.

Recordings with no audio on disk are skipped with a warning.


In [ ]:
def cents_from_hz(hz):
    return 1200.0 * np.log2(hz / 440.0) + 6900.0  # MIDI 69 = A4 = 6900 cents


def analyse_segment(
    rec_id: str, seg_row, audio_path: Path,
) -> tuple[metrics.ThreeAxisMetrics, pd.DataFrame] | None:
    seg_id = seg_row["segment_id"]

    # 1. Load segment audio.
    info = sf.info(str(audio_path))
    sr = info.samplerate
    start_frame = int(seg_row["start_s"] * sr)
    stop_frame = int(seg_row["end_s"] * sr)
    if stop_frame <= start_frame or stop_frame > info.frames:
        print(f"  [skip] {rec_id}/{seg_id}: segment outside audio bounds")
        return None
    y, _ = sf.read(str(audio_path), start=start_frame, stop=stop_frame)
    if y.ndim > 1:
        y = y.mean(axis=1)

    # 2. pYIN over the segment.
    track = pitch.track_pitch(
        y, sr,
        fmin=FMIN_HZ, fmax=FMAX_HZ,
        frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH,
    )

    # 3. Per-note diagnostics.
    seg_map = segment_score_map_df[
        (segment_score_map_df["recording_id"] == rec_id)
        & (segment_score_map_df["segment_id"] == seg_id)
    ].sort_values(["bar_number", "note_index"]).reset_index(drop=True)

    note_rows = []
    hop_s = HOP_LENGTH / sr
    for _, n in seg_map.iterrows():
        rel_start = float(n["start_s"]) - float(seg_row["start_s"])
        rel_end = float(n["end_s"]) - float(seg_row["start_s"])
        i0 = max(0, int(rel_start / hop_s))
        i1 = min(len(track.f0_hz), max(i0 + 1, int(rel_end / hop_s)))
        f0_slice = track.f0_hz[i0:i1]
        v_slice = track.voiced_flag[i0:i1]

        voiced_ratio = pitch.note_voiced_ratio(v_slice)
        f0_iqr = pitch.note_f0_iqr_cent(f0_slice, v_slice)

        f0_voiced = f0_slice[(v_slice > 0.5) & ~np.isnan(f0_slice) & (f0_slice > 0)]
        est_cent = float(np.median(cents_from_hz(f0_voiced))) if f0_voiced.size > 0 else np.nan

        score_match = score_events_df[
            (score_events_df["bar_number"] == int(n["bar_number"]))
            & (score_events_df["note_index"] == int(n["note_index"]))
        ]
        ref_cent = float(score_match.iloc[0]["ref_pitch_cent"]) if len(score_match) > 0 else np.nan
        err_cent = est_cent - ref_cent if not (np.isnan(est_cent) or np.isnan(ref_cent)) else np.nan

        pc_err = pitch.classify_pitch_class_error(err_cent)
        unreliable, reasons = pitch.is_pyin_unreliable(
            voiced_ratio=voiced_ratio,
            f0_iqr_cent=f0_iqr,
            pitch_class_error=pc_err,
        )

        note_rows.append({
            "recording_id": rec_id,
            "segment_id": seg_id,
            "bar_number": int(n["bar_number"]),
            "note_index": int(n["note_index"]),
            "est_cent": est_cent,
            "ref_cent": ref_cent,
            "err_cent": err_cent,
            "voiced_ratio": voiced_ratio,
            "f0_iqr_cent": f0_iqr,
            "pitch_class_error": pc_err,
            "is_pyin_unreliable": unreliable,
            "unreliable_reasons": reasons,
        })

    notes_df = pd.DataFrame(note_rows)
    if notes_df.empty:
        return None

    # 4. Three-axis metrics on the reliable subset.
    m = metrics.compute_three_axis_metrics(
        est_cent=notes_df["est_cent"].to_numpy(),
        score_cent=notes_df["ref_cent"].to_numpy(),
        unreliable_flags=notes_df["is_pyin_unreliable"].to_numpy(),
        min_notes=3,
    )
    return m, notes_df


# Drive the loop.
all_notes = []
segment_metrics = {}  # rec_id -> list[ThreeAxisMetrics]
missing_audio = []

for rec_id in RECORDINGS:
    audio_path = find_audio(rec_id)
    if audio_path is None:
        print(f"[skip] {rec_id}: no audio file found in {AUDIO_DIR}")
        missing_audio.append(rec_id)
        continue

    print(f"[run]  {rec_id}: using {audio_path.name}")
    rec_segments = segments_df[segments_df["recording_id"] == rec_id]
    if rec_segments.empty:
        print(f"  [skip] {rec_id}: no segments in metadata")
        continue

    metrics_list = []
    for _, seg_row in rec_segments.iterrows():
        result = analyse_segment(rec_id, seg_row, audio_path)
        if result is None:
            continue
        m, notes_df = result
        metrics_list.append(m)
        all_notes.append(notes_df)
        print(
            f"  {seg_row['segment_id']:25s} "
            f"offset={m.register_offset_cent:+7.1f}c  "
            f"range={m.range_compression:5.2f}  "
            f"contour={m.contour_correlation:+5.2f}  "
            f"n={m.n_notes_used}"
        )
    if metrics_list:
        segment_metrics[rec_id] = metrics_list

if missing_audio:
    print()
    print(f"Missing audio for: {missing_audio}")
    print("Place the corresponding files under data/audio/ to include them.")


## 5. Per-recording aggregation

`metrics.aggregate_metrics` reduces the per-segment list to four numbers
per recording: median register offset, median range compression, median
contour correlation, and the std of contour across segments (the
*dynamic* axis).


In [ ]:
agg_rows = []
for rec_id, metrics_list in segment_metrics.items():
    agg = metrics.aggregate_metrics(metrics_list)
    agg["recording_id"] = rec_id
    agg["n_segments"] = len(metrics_list)
    agg_rows.append(agg)

recording_summary = pd.DataFrame(agg_rows)[
    [
        "recording_id",
        "n_segments",
        "register_offset_cent",
        "range_compression",
        "contour_correlation_median",
        "contour_correlation_std",
    ]
]
print(recording_summary.to_string(index=False))

recording_summary.to_csv(EXPORT_DIR / "recording_summary.csv", index=False)
print(f"\nSaved: {EXPORT_DIR / 'recording_summary.csv'}")


## 6. Performance type classification

Decision flow (see `metrics.classify_performance` and
[`docs/method.md`](../docs/method.md) §3):

```
contour_std > 0.3          -> dynamic
|offset|    > 400 cents    -> directed-recitation
otherwise                   -> score-faithful
```


In [ ]:
recording_summary["performance_type"] = recording_summary.apply(
    lambda r: metrics.classify_performance(
        register_offset_cent=r["register_offset_cent"],
        contour_std=r["contour_correlation_std"],
    ),
    axis=1,
)
print(recording_summary[["recording_id", "performance_type"]].to_string(index=False))


## 7. Figure 1 — Radar chart

In [ ]:
recordings_data = {}
for _, r in recording_summary.iterrows():
    recordings_data[r["recording_id"]] = plotting.four_axes_normalize(
        register_offset_cent=r["register_offset_cent"],
        range_compression=r["range_compression"],
        contour_correlation=r["contour_correlation_median"],
        contour_std=r["contour_correlation_std"],
    )

if recordings_data:
    fig, _ = plotting.plot_radar_chart(recordings_data)
    fig.savefig(EXPORT_DIR / "fig1_radar.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {EXPORT_DIR / 'fig1_radar.png'}")
else:
    print("No recordings to plot.")


## 8. Figure 2 — PCA biplot (axis independence)

In [ ]:
if len(recordings_data) >= 3:
    fig, _ = plotting.plot_pca_biplot(recordings_data)
    fig.savefig(EXPORT_DIR / "fig2_pca_biplot.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {EXPORT_DIR / 'fig2_pca_biplot.png'}")
else:
    print(f"PCA biplot needs >=3 recordings; have {len(recordings_data)}.")


## 9. Figure 3 — Type classification flow

In [ ]:
flow_data = {
    r["recording_id"]: (r["register_offset_cent"], r["contour_correlation_std"])
    for _, r in recording_summary.iterrows()
}

if flow_data:
    fig, _ = plotting.plot_type_classification_flow(flow_data)
    fig.savefig(EXPORT_DIR / "fig3_type_flow.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {EXPORT_DIR / 'fig3_type_flow.png'}")


## 10. Per-note diagnostics export

For audit and downstream analysis we export every per-note diagnostic
(observed cents, score reference, error, voicing, IQR, the issue-#12
flag and its reasons) collected during the loop.


In [ ]:
if all_notes:
    notes_all_df = pd.concat(all_notes, ignore_index=True)
    notes_all_df.to_csv(EXPORT_DIR / "note_errors_all_segments.csv", index=False)
    print(f"Saved: {EXPORT_DIR / 'note_errors_all_segments.csv'}")
    n_unreliable = int(notes_all_df["is_pyin_unreliable"].sum())
    print(f"Total notes: {len(notes_all_df)}  unreliable: {n_unreliable}")
else:
    print("No per-note data collected.")


## Summary

This notebook is the reference implementation of the paper-1 batch
pipeline:

1. iterate over `(recording, segment)` pairs;
2. apply pitch tracking, reliability filtering, and three-axis metrics
   per segment;
3. aggregate to per-recording numbers;
4. classify each recording into one of three performance types;
5. produce the three core figures (radar, PCA biplot, decision flow);
6. export the per-recording summary and per-note diagnostics CSVs.

For the full set of paper-1 figures (Boulez 1961-1977 arrow,
her-1991 segment-level dynamics, summary table) extend the per-segment
loop above to retain segment-level metrics and add the matching
plotting helpers.

Numerical reproduction of the paper requires the five commercial
recordings listed at the top of this notebook; without them the
notebook will run on whatever subset is available, including the
single Stiedry-Wagner 1940 file fetched by `scripts/fetch_audio.py`.
